In [6]:
import serial
import time
import re

import serial.tools.list_ports

In [3]:
# === CONFIGURATION ===
COM_PORT = 'COM5'       # Replace with your actual COM port
BAUDRATE = 115200
EEPROM_ADDR = '0x50'    # I2C address of EEPROM
EEPROM_OFFSET = '0x00'  # Start reading from offset 0
BYTES_TO_READ = 16      # Number of bytes to read (must be <= 256 for U-Boot)

In [15]:
def send_cmd(cmd, wait_time=0.5):
    print(f"> {cmd}")
    ser.write((cmd + "\n").encode())
    time.sleep(wait_time)
    output = ser.read(ser.in_waiting).decode(errors='ignore')
    print(output)
    return output



# === PARSE OUTPUT ===
# Expected output format from U-Boot:
# 00: AB CD EF ...
def parse_i2c_md_output(output):
    data_bytes = []
    lines = output.splitlines()
    for line in lines:
        if ':' in line:
            parts = line.split(':', 1)
            hex_part = parts[1].strip()
            hex_bytes = hex_part.split()
            for b in hex_bytes:
                try:
                    data_bytes.append(int(b, 16))
                except ValueError:
                    continue
    return data_bytes

def try_probe(portname):
    try:
        ser = serial.Serial(portname, baudrate=115200, timeout=1)
        time.sleep(1)
        ser.write(b"i2c dev 0\n")
        time.sleep(0.5)
        ser.write(b"i2c probe\n")
        time.sleep(1)
        out = ser.read(ser.in_waiting).decode(errors='ignore')
        ser.close()
        if "50" in out or "0x5F" in out:
            return True, out
    except:
        pass
    return False, ""

In [7]:
ports = serial.tools.list_ports.comports()
print("Available COM ports:")
for port in ports:
    print(f"  {port.device} - {port.description}")

Available COM ports:
  COM8 - USB Serial Port (COM8)
  COM9 - USB Serial Port (COM9)
  COM40 - USB Serial Port (COM40)
  COM10 - USB Serial Port (COM10)
  COM41 - USB Serial Port (COM41)
  COM42 - USB Serial Port (COM42)
  COM12 - USB Serial Port (COM12)
  COM28 - USB Serial Port (COM28)
  COM13 - USB Serial Port (COM13)
  COM29 - USB Serial Port (COM29)
  COM14 - USB Serial Port (COM14)
  COM30 - USB Serial Port (COM30)


In [13]:
# ask user to choose a COM PORT

available = [(p.device, p.description) for p in ports]

print("Available COM ports:")
for i, (dev, desc) in enumerate(available):
    print(f"{i}: {dev} - {desc}")

# Ask user to pick
selection = int(input("Select the COM port index for your board: "))
com_port = available[selection][0]

print(f"\nUsing {com_port}...\n")

ser = serial.Serial(com_port, baudrate=115200, timeout=1)

Available COM ports:
0: COM8 - USB Serial Port (COM8)
1: COM9 - USB Serial Port (COM9)
2: COM40 - USB Serial Port (COM40)
3: COM10 - USB Serial Port (COM10)
4: COM41 - USB Serial Port (COM41)
5: COM42 - USB Serial Port (COM42)
6: COM12 - USB Serial Port (COM12)
7: COM28 - USB Serial Port (COM28)
8: COM13 - USB Serial Port (COM13)
9: COM29 - USB Serial Port (COM29)
10: COM14 - USB Serial Port (COM14)
11: COM30 - USB Serial Port (COM30)
Select the COM port index for your board: 0

Using COM8...



In [16]:
# Send i2c probe, check for response like 0x50 = EEPROM found.


ports = serial.tools.list_ports.comports()
for p in ports:
    found, output = try_probe(p.device)
    if found:
        print(f"✅ EEPROM found on {p.device} — response:\n{output}")
    else:
        print(f"❌ No EEPROM on {p.device}")

❌ No EEPROM on COM8
❌ No EEPROM on COM9
❌ No EEPROM on COM40
❌ No EEPROM on COM10
❌ No EEPROM on COM41
❌ No EEPROM on COM42
❌ No EEPROM on COM12
❌ No EEPROM on COM28
❌ No EEPROM on COM13
❌ No EEPROM on COM29
❌ No EEPROM on COM14
❌ No EEPROM on COM30


In [5]:
# === SETUP SERIAL PORT ===
ser = serial.Serial(COM_PORT, baudrate=BAUDRATE, timeout=1)
time.sleep(2)  # Give U-Boot time to start

SerialException: could not open port 'COM5': FileNotFoundError(2, 'O sistema não pode encontrar o arquivo especificado.', None, 2)

In [ ]:
# === I2C READ SEQUENCE IN U-BOOT ===
send_cmd("i2c dev 0")
send_cmd("i2c probe")

In [ ]:
# Read EEPROM bytes using 'i2c md' (memory display)
# Syntax: i2c md <chip> <addr> <length>
read_output = send_cmd(f"i2c md {EEPROM_ADDR} {EEPROM_OFFSET} {BYTES_TO_READ}", wait_time=1.0)


eeprom_data = parse_i2c_md_output(read_output)

# === DISPLAY RESULT ===
print("\nEEPROM Raw Bytes:")
print(' '.join(f'{b:02X}' for b in eeprom_data))